In [1]:
import pandas as pd

df = pd.read_parquet("data/crimes_raw.parquet")
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df.head()

Loaded 1,500,000 rows, 22 columns


,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,ward,community_area,fbi_code,year,updated_on,x_coordinate,y_coordinate,latitude,longitude,location
0,13437810,JH235640,2019-01-01T00:00:00.000,056XX S GREEN ST,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,APARTMENT,False,False,...,16,68,11,2019,2024-04-23T15:41:34.000,NaN,NaN,NaN,NaN,None
1,13442554,JH240629,2019-01-01T00:00:00.000,081XX S COTTAGE GROVE AVE,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,NaN,False,False,...,6,44,11,2019,2024-04-28T15:41:07.000,NaN,NaN,NaN,NaN,None
2,13448893,JH249388,2019-01-01T00:00:00.000,016XX W SHERWIN AVE,1582,OFFENSE INVOLVING CHILDREN,CHILD PORNOGRAPHY,APARTMENT,False,True,...,49,1,17,2019,2024-05-04T15:52:16.000,NaN,NaN,NaN,NaN,None
3,13290715,JG520302,2019-01-01T00:00:00.000,076XX S MORGAN ST,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,APARTMENT,False,False,...,17,71,11,2019,2023-11-28T15:41:41.000,NaN,NaN,NaN,NaN,None
4,13256724,JG476831,2019-01-01T00:00:00.000,034XX W SUNNYSIDE AVE,1562,SEX OFFENSE,AGGRAVATED CRIMINAL SEXUAL ABUSE,RESIDENCE,True,True,...,33,14,17,2019,2024-03-10T15:40:42.000,NaN,NaN,NaN,NaN,None


In [2]:
df.dtypes

id                         str
case_number                str
date                       str
block                      str
iucr                       str
primary_type               str
description                str
location_description       str
arrest                    bool
domestic                  bool
beat                       str
district                   str
ward                       str
community_area             str
fbi_code                   str
year                       str
updated_on                 str
x_coordinate               str
y_coordinate               str
latitude                   str
longitude                  str
location                object
dtype: object

In [3]:
# parse the date string into a real datetime
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# derive time features for the temporal analysis
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["hour"] = df["date"].dt.hour
df["dayofweek"] = df["date"].dt.day_name()       # "Monday", "Tuesday", ...
df["month_name"] = df["date"].dt.month_name()    # "January", ... (handy for viz)

# check it worked
print(df["date"].dtype)
print(df[["date", "year", "month", "hour", "dayofweek"]].head())

datetime64[us]
        date  year  month  hour dayofweek
0 2019-01-01  2019      1     0   Tuesday
1 2019-01-01  2019      1     0   Tuesday
2 2019-01-01  2019      1     0   Tuesday
3 2019-01-01  2019      1     0   Tuesday
4 2019-01-01  2019      1     0   Tuesday


In [4]:
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

before = len(df)
df = df.dropna(subset=["latitude", "longitude"])
after = len(df)
print(f"Dropped {before - after:,} rows missing coordinates")
print(f"{after:,} rows remain")

Dropped 23,819 rows missing coordinates
1,476,181 rows remain


In [5]:
df["community_area"] = pd.to_numeric(df["community_area"], errors="coerce")

before = len(df)
df = df.dropna(subset=["community_area"])
df["community_area"] = df["community_area"].astype(int)
print(f"Dropped {before - len(df):,} rows missing community area")
print(f"{df['community_area'].nunique()} unique community areas (should be ~77)")

Dropped 141 rows missing community area
77 unique community areas (should be ~77)


In [6]:
print(df["primary_type"].nunique(), "unique crime types\n")
print(df["primary_type"].value_counts())

33 unique crime types

primary_type
THEFT                                324478
BATTERY                              270929
CRIMINAL DAMAGE                      166808
ASSAULT                              130201
MOTOR VEHICLE THEFT                  105142
DECEPTIVE PRACTICE                   102702
OTHER OFFENSE                         93803
ROBBERY                               54116
WEAPONS VIOLATION                     50176
BURGLARY                              49979
NARCOTICS                             43735
CRIMINAL TRESPASS                     29357
OFFENSE INVOLVING CHILDREN            11420
CRIMINAL SEXUAL ASSAULT                8196
SEX OFFENSE                            7202
PUBLIC PEACE VIOLATION                 6150
INTERFERENCE WITH PUBLIC OFFICER       4408
HOMICIDE                               4189
ARSON                                  2986
STALKING                               2339
PROSTITUTION                           1881
CONCEALED CARRY LICENSE VIOLATION      1

In [8]:
crime_category_map = {
    # ---- Violent ----
    "BATTERY": "Violent",
    "ASSAULT": "Violent",
    "ROBBERY": "Violent",
    "HOMICIDE": "Violent",
    "CRIMINAL SEXUAL ASSAULT": "Violent",
    "SEX OFFENSE": "Violent",
    "CRIM SEXUAL ASSAULT": "Violent",      # older label, just in case
    "KIDNAPPING": "Violent",
    "HUMAN TRAFFICKING": "Violent",
    "INTIMIDATION": "Violent",
    "STALKING": "Violent",
    "OFFENSE INVOLVING CHILDREN": "Violent",

    # ---- Property ----
    "THEFT": "Property",
    "CRIMINAL DAMAGE": "Property",
    "MOTOR VEHICLE THEFT": "Property",
    "BURGLARY": "Property",
    "ARSON": "Property",
    "CRIMINAL TRESPASS": "Property",

    # ---- Quality-of-Life ----
    "NARCOTICS": "Quality-of-Life",
    "WEAPONS VIOLATION": "Quality-of-Life",
    "PUBLIC PEACE VIOLATION": "Quality-of-Life",
    "PROSTITUTION": "Quality-of-Life",
    "LIQUOR LAW VIOLATION": "Quality-of-Life",
    "GAMBLING": "Quality-of-Life",
    "INTERFERENCE WITH PUBLIC OFFICER": "Quality-of-Life",
    "CONCEALED CARRY LICENSE VIOLATION": "Quality-of-Life",
    "OTHER NARCOTIC VIOLATION": "Quality-of-Life",
    "PUBLIC INDECENCY": "Quality-of-Life",
    "OBSCENITY": "Quality-of-Life",

    # ---- Other ----
    "DECEPTIVE PRACTICE": "Other",
    "OTHER OFFENSE": "Other",
    "NON-CRIMINAL": "Other",
    "NON - CRIMINAL": "Other",
    "NON-CRIMINAL (SUBJECT SPECIFIED)": "Other",
    "RITUALISM": "Other",
}

df["crime_category"] = df["primary_type"].map(crime_category_map)

# safety net: anything not in the map shows up here
unmapped = df[df["crime_category"].isna()]["primary_type"].value_counts()
if len(unmapped) > 0:
    print("⚠️ Unmapped types (need to add these):")
    print(unmapped)
else:
    print("✅ All crime types mapped")

print("\nCategory breakdown:")
print(df["crime_category"].value_counts())

✅ All crime types mapped

Category breakdown:
crime_category
Property           678750
Violent            491415
Other              196527
Quality-of-Life    109348
Name: count, dtype: int64


In [9]:
keep_cols = [
    "id", "date", "year", "month", "month_name", "hour", "dayofweek",
    "primary_type", "crime_category", "description",
    "arrest", "domestic",
    "community_area", "district", "ward",
    "latitude", "longitude", "location_description"
]
clean = df[keep_cols].copy()

print(f"Clean dataset: {len(clean):,} rows, {clean.shape[1]} columns")
clean.head()

Clean dataset: 1,476,040 rows, 18 columns


,id,date,year,month,month_name,hour,dayofweek,primary_type,crime_category,description,arrest,domestic,community_area,district,ward,latitude,longitude,location_description
9,13187275,2019-01-01,2019,1,January,0,Tuesday,CRIMINAL SEXUAL ASSAULT,Violent,AGGRAVATED - KNIFE / CUTTING INSTRUMENT,True,False,66,008,15,41.771731,-87.698847,RESIDENCE
10,13187280,2019-01-01,2019,1,January,0,Tuesday,CRIMINAL SEXUAL ASSAULT,Violent,AGGRAVATED - KNIFE / CUTTING INSTRUMENT,True,False,66,008,15,41.771731,-87.698847,RESIDENCE
25,11552758,2019-01-01,2019,1,January,0,Tuesday,CRIMINAL DAMAGE,Property,TO PROPERTY,False,False,67,007,16,41.778565,-87.665464,APARTMENT
26,11571975,2019-01-01,2019,1,January,0,Tuesday,DECEPTIVE PRACTICE,Other,FINANCIAL IDENTITY THEFT OVER $ 300,False,False,25,015,28,41.880925,-87.757528,RESIDENCE
27,11552709,2019-01-01,2019,1,January,0,Tuesday,BATTERY,Violent,DOMESTIC BATTERY SIMPLE,False,True,58,009,15,41.812780,-87.691894,APARTMENT


In [10]:
import os
clean.to_parquet("data/crimes_clean.parquet", index=False)

size_mb = os.path.getsize("data/crimes_clean.parquet") / 1e6
print(f"Saved data/crimes_clean.parquet ({size_mb:.1f} MB)")
print(os.listdir("data"))

Saved data/crimes_clean.parquet (42.3 MB)
['crimes_raw.parquet', 'crimes_clean.parquet']
